In [111]:
import mysql.connector

con = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Jasmi@123"
)

cursor = con.cursor()

print("Connected Successfully")

Connected Successfully


In [113]:
cursor.execute("CREATE DATABASE IF NOT EXISTS inventory_db")
print("Database Ready")

Database Ready


In [115]:
cursor.execute("USE inventory_db")
print("Database Selected")

Database Selected


In [117]:
cursor.execute("SHOW DATABASES")

for db in cursor:
    print(db)

('django31',)
('information_schema',)
('inventory_db',)
('jasmi23',)
('jasmitha23',)
('mysql',)
('performance_schema',)
('sakila',)
('sys',)
('world',)


In [119]:
cursor.execute("SELECT DATABASE()")

print(cursor.fetchone())

('inventory_db',)


In [121]:
query = """
CREATE TABLE IF NOT EXISTS products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(50),
    cost_price DECIMAL(10,2),
    sell_price DECIMAL(10,2),
    quantity INT
)
"""

cursor.execute(query)
con.commit()

print("Products table created successfully.")

Products table created successfully.


In [123]:
cursor.execute("SHOW TABLES")

for table in cursor:
    print(table)

('orders',)
('owner',)
('products',)
('users',)


In [125]:
query = """
CREATE TABLE IF NOT EXISTS owner(
    username VARCHAR(30) PRIMARY KEY,
    password VARCHAR(30)
)
"""

cursor.execute(query)
con.commit()

print("Owner table created successfully")

Owner table created successfully


In [127]:
query = """
CREATE TABLE IF NOT EXISTS users(
    username VARCHAR(30) PRIMARY KEY,
    password VARCHAR(30),
    phone VARCHAR(10)
)
"""

cursor.execute(query)
con.commit()

print("Users table created successfully")

Users table created successfully


In [129]:
query = """
CREATE TABLE IF NOT EXISTS orders(
    order_id INT AUTO_INCREMENT PRIMARY KEY,
    username VARCHAR(30),
    total_amount DECIMAL(10,2),
    order_date DATETIME DEFAULT CURRENT_TIMESTAMP
)
"""

cursor.execute(query)
con.commit()

print("Orders table created successfully")

Orders table created successfully


In [131]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table)

('orders',)
('owner',)
('products',)
('users',)


In [133]:
query = """
INSERT IGNORE INTO owner(username, password)
VALUES('admin', 'admin123')
"""

cursor.execute(query)
con.commit()

In [135]:
cursor.execute("SELECT * FROM owner")

for row in cursor.fetchall():
    print(row)

('admin', 'admin123')


In [137]:
import datetime

In [139]:
def owner_login():

    print("\n========== OWNER LOGIN ==========")

    attempts = 3

    while attempts > 0:

        username = input("Enter Username: ").strip()
        password = input("Enter Password: ").strip()

        if username == "" or password == "":
            print("\n Username and Password cannot be empty.")
            continue

        query = """
        SELECT * FROM owner
        WHERE username=%s AND password=%s
        """

        cursor.execute(query, (username, password))
        data = cursor.fetchone()

        if data:
            print("\n Login Successful")
            owner_menu()
            return

        else:
            attempts -= 1

            if attempts > 0:
                print(f"\n Invalid Username or Password.")
                print(f"Remaining Attempts: {attempts}")

            else:
                print("\n Login Failed!")
                print("You have exceeded the maximum login attempts.")
                return

In [141]:
def add_product():

    try:

        product_id = int(input("Enter Product ID: "))
        product_name = input("Enter Product Name: ").strip().title()
        cost_price = float(input("Enter Cost Price: "))
        sell_price = float(input("Enter Sell Price: "))
        quantity = int(input("Enter Quantity: "))

        query = """
        INSERT INTO products
        VALUES(%s,%s,%s,%s,%s)
        """

        values = (
            product_id,
            product_name,
            cost_price,
            sell_price,
            quantity
        )

        cursor.execute(query, values)
        con.commit()

        print("\n Product Added Successfully")

    except mysql.connector.IntegrityError:
        print("\n Product ID already exists.")

    except ValueError:
        print("\n Please enter valid numeric values.")

    except Exception as e:
        print("\nError:", e)

In [143]:
def view_products():

    cursor.execute("SELECT * FROM products")
    products = cursor.fetchall()

    print("\nProduct List")
    print("-" * 65)

    print(f"{'ID':<8}{'Name':<15}{'Cost':<10}{'Sell':<10}{'Qty':<10}")

    print("-" * 65)

    for product in products:
        print(f"{product[0]:<8}{product[1]:<15}{product[2]:<10.2f}{product[3]:<10.2f}{product[4]:<10}")

    print("-" * 65)

In [145]:
def update_product():

    try:

        pid = int(input("Enter Product ID: "))

        cursor.execute("SELECT * FROM products WHERE product_id=%s", (pid,))
        data = cursor.fetchone()

        if data is None:
            print(" Product not found.")
            return

        product_name = input("Enter New Product Name: ").strip().title()
        cost_price = float(input("Enter New Cost Price: "))
        sell_price = float(input("Enter New Selling Price: "))
        quantity = int(input("Enter New Quantity: "))

        query = """
        UPDATE products
        SET product_name=%s,
            cost_price=%s,
            sell_price=%s,
            quantity=%s
        WHERE product_id=%s
        """

        values = (
            product_name,
            cost_price,
            sell_price,
            quantity,
            pid
        )

        cursor.execute(query, values)
        con.commit()

        print("Product Updated Successfully")

    except ValueError:
        print(" Please enter valid input.")

    except Exception as e:
        print("Error:", e)

In [147]:
def delete_product():

    try:

        pid = int(input("Enter Product ID: "))

        query = "DELETE FROM products WHERE product_id=%s"

        cursor.execute(query, (pid,))
        con.commit()

        if cursor.rowcount > 0:
            print("Product Deleted Successfully")
        else:
            print("Product ID not found.")

    except ValueError:
        print("Enter valid Product ID.")

    except Exception as e:
        print("Error:", e)

In [149]:
def search_product():

    name = input("Enter Product Name: ").strip().title()

    query = "SELECT * FROM products WHERE product_name=%s"

    cursor.execute(query, (name,))

    data = cursor.fetchone()

    if data:
        print("\nProduct Found")
        print("-" * 60)
        print("ID :", data[0])
        print("Name :", data[1])
        print("Cost Price :", data[2])
        print("Selling Price :", data[3])
        print("Quantity :", data[4])
    else:
        print(" Product not found.")

In [151]:
def owner_menu():

    while True:

        print("\n" + "=" * 40)
        print("              OWNER MENU")
        print("=" * 40)
        print("1. Add Product")
        print("2. View Products")
        print("3. Update Product")
        print("4. Delete Product")
        print("5. Search Product")
        print("6. Logout")
        print("=" * 40)

        choice = input("Enter Choice (1-6): ").strip()

        # Validate input
        if not choice.isdigit():
            print("\n Invalid Input! Please enter numbers only.")
            continue

        choice = int(choice)

        if choice == 1:
            add_product()

        elif choice == 2:
            view_products()

        elif choice == 3:
            update_product()

        elif choice == 4:
            delete_product()

        elif choice == 5:
            search_product()

        elif choice == 6:
            print("\n Logout Successful")
            break

        else:
            print("\n Invalid Choice! Please enter a number between 1 and 6.")

In [153]:
def main_menu():

    while True:

        print("\n" + "=" * 45)
        print("         INVENTORY MANAGEMENT SYSTEM")
        print("=" * 45)
        print("1. Owner Login")
        print("2. User Registration")
        print("3. User Login")
        print("4. Exit")
        print("=" * 45)

        choice = input("Enter Choice (1-4): ").strip()

        # Check whether input is a number
        if not choice.isdigit():
            print("\n Invalid Input! Please enter numbers only (1-4).")
            continue

        choice = int(choice)

        if choice == 1:
            owner_login()

        elif choice == 2:
            register_user()

        elif choice == 3:
            user_login()

        elif choice == 4:
            print("\n Thank You for using Inventory Management System.")
            print("Goodbye!")
            break

        else:
            print("\n Invalid Choice! Please enter a number between 1 and 4.")

In [155]:
def register_user():

    print("\n========== USER REGISTRATION ==========")

    name = input("Enter Name: ")

    while True:
        phone = input("Enter Phone Number: ")

        if len(phone) == 10 and phone.isdigit() and phone[0] in ["9", "8", "7", "6"]:
            break
        else:
            print(" Invalid Phone Number")
            print("Phone number must be 10 digits and start with 9, 8, 7, or 6.")

    username = input("Create Username: ")
    password = input("Create Password: ")

    # Check Username
    cursor.execute(
        "SELECT * FROM users WHERE username=%s",
        (username,)
    )

    user = cursor.fetchone()

    if user:
        print("\n Username already exists.")
        print("Please choose another username.")
        return

    query = """
    INSERT INTO users(name, phone, username, password)
    VALUES(%s,%s,%s,%s)
    """

    values = (name, phone, username, password)

    cursor.execute(query, values)
    con.commit()

    print("\n Registration Successful")

In [157]:
query = """
CREATE TABLE IF NOT EXISTS users(
    user_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50),
    phone VARCHAR(15),
    username VARCHAR(30) UNIQUE,
    password VARCHAR(30)
)
"""

cursor.execute(query)
con.commit()

print("Users table created successfully.")

Users table created successfully.


In [159]:
def user_login():

    print("\n========== USER LOGIN ==========")

    attempts = 3

    while attempts > 0:

        username = input("Enter Username: ").strip()
        password = input("Enter Password: ").strip()

        if username == "" or password == "":
            print("\n Username and Password cannot be empty.")
            continue

        query = """
        SELECT * FROM users
        WHERE username=%s AND password=%s
        """

        cursor.execute(query, (username, password))
        user = cursor.fetchone()

        if user:
            print("\n Login Successful")
            user_menu(user[0])
            return

        else:
            attempts -= 1

            if attempts > 0:
                print("\n Invalid Username or Password.")
                print(f"Remaining Attempts: {attempts}")

            else:
                print("\n Login Failed!")
                print("You have exceeded the maximum login attempts.")
                return

In [161]:
def user_view_products():

    cursor.execute("""
        SELECT product_id,
               product_name,
               sell_price,
               quantity
        FROM products
    """)

    products = cursor.fetchall()

    if not products:
        print("\n No Products Available.")
        return

    print("\n" + "=" * 60)
    print("                AVAILABLE PRODUCTS")
    print("=" * 60)

    print(f"{'ID':<8}{'Product Name':<20}{'Price':<12}{'Quantity'}")

    print("-" * 60)

    for product in products:
        print(f"{product[0]:<8}{product[1]:<20}{product[2]:<12.2f}{product[3]}")

    print("=" * 60)

In [163]:
def user_menu(user_id):

    while True:

        print("\n" + "=" * 40)
        print("               USER MENU")
        print("=" * 40)
        print("1. View Products")
        print("2. Buy Product")
        print("3. Logout")
        print("=" * 40)

        choice = input("Enter Choice (1-3): ").strip()

        if not choice.isdigit():
            print("\n Please enter numbers only.")
            continue

        choice = int(choice)

        if choice == 1:
            user_view_products()

        elif choice == 2:
            buy_product(user_id)

        elif choice == 3:
            print("\n Logout Successful")
            break

        else:
            print("\n Invalid Choice! Please enter a number between 1 and 3.")

In [165]:
def buy_product(user_id):

    cart = []

    # Customer Details
    cursor.execute(
        "SELECT name, phone, username FROM users WHERE user_id=%s",
        (user_id,)
    )
    customer = cursor.fetchone()

    while True:

        print("\n========== BUY PRODUCT ==========")

        user_view_products()

        # Product ID
        product_id = input("\nEnter Product ID: ").strip()

        if not product_id.isdigit():
            print("\n1 Product ID must be a number.")
            continue

        product_id = int(product_id)

        # Quantity
        qty = input("Enter Quantity: ").strip()

        if not qty.isdigit():
            print("\n Quantity must be a number.")
            continue

        qty = int(qty)

        if qty <= 0:
            print("\n Quantity must be greater than zero.")
            continue

        # Product Details
        cursor.execute(
            "SELECT * FROM products WHERE product_id=%s",
            (product_id,)
        )

        product = cursor.fetchone()

        if product is None:
            print("\n Product Not Found.")
            continue

        if qty > product[4]:
            print(f"\n Only {product[4]} items available in stock.")
            continue

        sell_price = product[3]
        amount = sell_price * qty

        # Update Stock
        new_qty = product[4] - qty

        cursor.execute(
            "UPDATE products SET quantity=%s WHERE product_id=%s",
            (new_qty, product_id)
        )

        # Save Order
        cursor.execute(
            "INSERT INTO orders(username,total_amount) VALUES(%s,%s)",
            (customer[2], amount)
        )

        con.commit()

        # Add to Cart
        cart.append({
            "product": product,
            "qty": qty,
            "amount": amount
        })

        print("\n Product Added Successfully!")

        while True:

            print("\nDo you want to continue shopping?")
            print("1. Yes")
            print("2. No")

            option = input("Enter Choice: ").strip()

            if option == "1":
                break

            elif option == "2":

                generate_bill(customer, cart)

                print("\n🛒 Thank You for Shopping with Fresh Mart!")
                return

            else:
                print("\n Invalid Choice! Please enter 1 or 2.")

In [167]:
def generate_bill(customer, cart):

    import datetime

    now = datetime.datetime.now()

    grand_total = 0

    print("\n")
    print("=" * 65)
    print("                 FRESH MART INVENTORY")
    print("=" * 65)
    print("Bill No   :", now.strftime("%H%M%S"))
    print("Date      :", now.strftime("%d-%m-%Y"))
    print("Time      :", now.strftime("%I:%M %p"))
    print("-" * 65)
    print("Customer  :", customer[0])
    print("Phone     :", customer[1])
    print("-" * 65)

    print(f"{'Product':<20}{'Qty':<10}{'Price':<12}{'Amount'}")
    print("-" * 65)

    for item in cart:

        product = item["product"]
        qty = item["qty"]
        amount = item["amount"]

        grand_total += amount

        print(f"{product[1]:<20}{qty:<10}{product[3]:<12.2f}{amount:.2f}")

    print("-" * 65)
    print(f"{'Grand Total':<42}{grand_total:.2f}")
    print("=" * 65)
    print("          Thank You! Visit Again ")
    print("=" * 65)

In [169]:
def view_orders():

    cursor.execute("SELECT * FROM orders")

    orders = cursor.fetchall()

    if not orders:
        print("\nNo Orders Found.")
        return

    print("\n" + "=" * 65)
    print("                    ORDER HISTORY")
    print("=" * 65)

    print(f"{'Order ID':<10}{'Username':<20}{'Amount':<15}{'Date'}")

    print("-" * 65)

    for order in orders:
        print(f"{order[0]:<10}{order[1]:<20}{order[2]:<15.2f}{order[3]}")

    print("=" * 65)

In [171]:
def sales_report():

    query = """
    SELECT SUM(total_amount)
    FROM orders
    """

    cursor.execute(query)

    total = cursor.fetchone()[0]

    if total is None:
        total = 0

    print("\n" + "=" * 40)
    print("          SALES REPORT")
    print("=" * 40)

    print(f"Total Sales : ₹ {total:.2f}")

    print("=" * 40)

In [173]:
def owner_menu():

    while True:

        print("\n" + "=" * 45)
        print("              OWNER MENU")
        print("=" * 45)
        print("1. Add Product")
        print("2. View Products")
        print("3. Update Product")
        print("4. Delete Product")
        print("5. Search Product")
        print("6. View Orders")
        print("7. Sales Report")
        print("8. Logout")
        print("=" * 45)

        choice = input("Enter Choice (1-8): ").strip()

        if not choice.isdigit():
            print("\n Please enter numbers only.")
            continue

        choice = int(choice)

        if choice == 1:
            add_product()

        elif choice == 2:
            view_products()

        elif choice == 3:
            update_product()

        elif choice == 4:
            delete_product()

        elif choice == 5:
            search_product()

        elif choice == 6:
            view_orders()

        elif choice == 7:
            sales_report()

        elif choice == 8:
            print("\n Logout Successful")
            break

        else:
            print("\n Invalid Choice! Please enter a number between 1 and 8.")

In [175]:
main_menu()


         INVENTORY MANAGEMENT SYSTEM
1. Owner Login
2. User Registration
3. User Login
4. Exit


Enter Choice (1-4):  1



========== OWNER LOGIN ==========


Enter Username:  admin
Enter Password:  admin123



 Login Successful

              OWNER MENU
1. Add Product
2. View Products
3. Update Product
4. Delete Product
5. Search Product
6. View Orders
7. Sales Report
8. Logout


...


          SALES REPORT
Total Sales : ₹ 0.00

              OWNER MENU
1. Add Product
2. View Products
3. Update Product
4. Delete Product
5. Search Product
6. View Orders
7. Sales Report
8. Logout


Enter Choice (1-8):  8



 Logout Successful

         INVENTORY MANAGEMENT SYSTEM
1. Owner Login
2. User Registration
3. User Login
4. Exit


Enter Choice (1-4):  4



 Thank You for using Inventory Management System.
Goodbye!


In [85]:
import mysql.connector

con = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Jasmi@123"
)

cursor = con.cursor()

In [87]:
cursor.execute("USE inventory_db")   # Replace with your actual database name

In [89]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table)

('orders',)
('owner',)
('products',)
('users',)


In [91]:
cursor.execute("SELECT * FROM users")
print(cursor.fetchall())

[(1, 'jasmi', '9877654468', 'jasmi', '1234')]


In [93]:
cursor.execute("SELECT * FROM products")
print(cursor.fetchall())

[(101, 'Tomato', Decimal('12.00'), Decimal('20.00'), 98), (102, 'Onion', Decimal('25.00'), Decimal('35.00'), 80), (103, 'Potato', Decimal('25.00'), Decimal('30.00'), 48)]


In [95]:
cursor.execute("SELECT * FROM orders")
print(cursor.fetchall())

[(1, 'jasmi', Decimal('168.00'), datetime.datetime(2026, 7, 7, 20, 57, 7)), (2, 'jasmi', Decimal('6084.00'), datetime.datetime(2026, 7, 7, 20, 57, 26)), (3, 'jasmi', Decimal('112.00'), datetime.datetime(2026, 7, 7, 21, 18, 2)), (4, 'jasmi', Decimal('546.00'), datetime.datetime(2026, 7, 7, 21, 18, 19)), (5, 'jasmi', Decimal('30.00'), datetime.datetime(2026, 7, 7, 21, 22, 10)), (6, 'jasmi', Decimal('60.00'), datetime.datetime(2026, 7, 7, 21, 26, 9)), (7, 'jasmi', Decimal('60.00'), datetime.datetime(2026, 7, 9, 21, 42, 53)), (8, 'jasmi', Decimal('40.00'), datetime.datetime(2026, 7, 10, 11, 25, 4)), (9, 'jasmi', Decimal('210.00'), datetime.datetime(2026, 7, 14, 21, 7, 30)), (10, 'jasmi', Decimal('60.00'), datetime.datetime(2026, 7, 14, 21, 7, 44)), (11, 'jasmi', Decimal('240.00'), datetime.datetime(2026, 7, 14, 21, 8, 4))]


In [97]:
cursor.execute("DELETE FROM orders")
cursor.execute("DELETE FROM users")
cursor.execute("DELETE FROM products")

con.commit()

print("Dummy data deleted successfully!")

Dummy data deleted successfully!


In [103]:
cursor.execute("SELECT * FROM orders")
print(cursor.fetchall())

[]


In [105]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table)

('orders',)
('owner',)
('products',)
('users',)


In [107]:
cursor.execute("DELETE FROM orders")
con.commit()

print("Orders deleted successfully.")

Orders deleted successfully.


In [109]:
cursor.execute("SELECT * FROM users")
print(cursor.fetchall())

cursor.execute("SELECT * FROM orders")
print(cursor.fetchall())

cursor.execute("SELECT * FROM products")
print(cursor.fetchall())

[]
[]
[]


In [177]:
import os
print(os.getcwd())

C:\Users\murak


In [179]:
import os

print(os.getcwd())

C:\Users\murak
